# 04. Ejecutar experimentos


Se usaron 29,965 clientes y 20 variables predictoras, con 22.13 % de *default*. Ya no había duplicados por eliminar y se mantienen los 21 grupos con variables idénticas y etiqueta distinta (ruido del dataset). Se ejecutaron 96 combinaciones (6 modelos × 4 balanceos × 4 métodos de búsqueda de hiperparámetros) con validación cruzada anidada y el F1 como métrica de selección. La SVM se corrió aparte en Colab con GPU y completa las 112 combinaciones.

- **El balanceo es lo que más influye.** `class_weight` obtuvo el mayor F1 en los tres modelos (Random Forest 0.544 a 0.546, XGBoost 0.544 a 0.545 y KNN 0.524 a 0.527). SMOTE y ADASYN quedaron en un nivel intermedio y sin balanceo el F1 fue el más bajo (por ejemplo, 0.473 a 0.479 en Random Forest).
- **El método de búsqueda importa poco en F1, pero mucho en tiempo.** Con `class_weight` las diferencias entre métodos son menores que la desviación estándar. En cambio, el algoritmo genético tardó mucho más: 2,432 s frente a 178 s con búsqueda en malla en Random Forest, y hasta 10,880 s en KNN sin balanceo.
- **Mejor configuración:** Random Forest con `class_weight` y búsqueda genética, con F1 de 0.5460 ± 0.0088, prácticamente empatada con otras cuatro.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np

from src import config
from src.data_loader import load_processed_dataset, split_features_target
from src.synthetic_data import generate_synthetic_dataset
from src.preprocessing import remove_duplicate_rows, check_label_conflicts
from src.models import MODEL_REGISTRY
from src.experiment_runner import run_experiments, get_all_combinations, MODEL_COST_ORDER
from src.evaluation import format_metrics_table

np.random.seed(config.RANDOM_STATE)

C:\Users\valef\anaconda3\envs\creditcard\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Cargar datos



In [2]:
try:
    df = load_processed_dataset()
    print(f"Dataset real cargado: {df.shape}")
except FileNotFoundError as e:
    print("Dataset real aun no disponible, usando datos sinteticos para desarrollo/pruebas:")
    print(f"  {e}")
    df = generate_synthetic_dataset()
    print(f"Dataset sintetico generado: {df.shape}")

df, n_removed = remove_duplicate_rows(df)
print(f"Filas duplicadas eliminadas: {n_removed}")

n_conflicts = check_label_conflicts(df)
if n_conflicts > 0:
    print(
        f"AVISO: {n_conflicts} grupos de filas con features identicas pero target distinto "
        "(ruido irreducible del dataset, no se resuelve automaticamente)."
    )

X, y = split_features_target(df)
print(f"X: {X.shape}, balance del target: {y.value_counts(normalize=True).round(4).to_dict()}")

Dataset real cargado: (29965, 21)
Filas duplicadas eliminadas: 0
AVISO: 21 grupos de filas con features identicas pero target distinto (ruido irreducible del dataset, no se resuelve automaticamente).
X: (29965, 20), balance del target: {0: 0.7787, 1: 0.2213}


## Configuración de la corrida



In [3]:
# Todos los modelos menos SVM (se corre en Colab con GPU, ver 07_svm_gpu_colab.ipynb).
models = [spec for name, spec in MODEL_REGISTRY.items() if name != "svm"]

# Las 4 técnicas de balanceo y los 4 métodos de optimización (None = todos).
techniques = None
methods = None
n_outer_folds, n_inner_folds = config.N_OUTER_FOLDS, config.N_INNER_FOLDS

# n_jobs por método (ver config.OPTIMIZATION_N_JOBS): paraleliza el eje de
# evaluaciones independientes de CADA método (candidatos de grid/random
# search, trials de Optuna, individuos de la población en genetic_deap).
# No cambia folds/rangos/tamaño de población/iteraciones, solo el tiempo de
# pared -- ver EXPERIMENT_STATUS.md sección "Paralelismo (2026-09-22)".
optimizer_kwargs_by_method = {
    "grid_search": {"n_jobs": config.OPTIMIZATION_N_JOBS},
    "random_search": {"n_iter": config.RANDOM_SEARCH_ITER, "n_jobs": config.OPTIMIZATION_N_JOBS},
    "bayesian_optuna": {"n_trials": config.OPTUNA_N_TRIALS, "n_jobs": config.OPTIMIZATION_N_JOBS},
    "genetic_deap": {
        "population_size": config.GA_POPULATION_SIZE,
        "n_generations": config.GA_N_GENERATIONS,
        "n_jobs": config.OPTIMIZATION_N_JOBS,
    },
}
# n_jobs=1 aquí (bucle EXTERNO sobre combinaciones, secuencial): el
# paralelismo real vive dentro de cada método, para no anidar paralelismo.
n_jobs = 1

# timeout_seconds=None: desactivado (ver EXPERIMENT_STATUS.md sección 8d).
timeout_seconds = None

combos = get_all_combinations(models, techniques, methods)
print(f"{len(combos)} combinaciones a considerar.")
print(f"Orden por costo computacional: {MODEL_COST_ORDER}")

96 combinaciones a considerar.
Orden por costo computacional: ('naive_bayes', 'logistic_regression', 'decision_tree', 'xgboost', 'random_forest', 'knn', 'svm')


In [4]:
table, results = run_experiments(
    X,
    y,
    models=models,
    techniques=techniques,
    methods=methods,
    n_outer_folds=n_outer_folds,
    n_inner_folds=n_inner_folds,
    optimizer_kwargs_by_method=optimizer_kwargs_by_method,
    checkpoint_path=config.EXPERIMENTS_MASTER_PATH,
    resume=True,
    n_jobs=n_jobs,
    timeout_seconds=timeout_seconds,
)

print(f"\n{len(results)} combinaciones nuevas corridas en esta sesion.")
print(f"{len(table)} filas totales en la tabla maestra ({config.EXPERIMENTS_MASTER_PATH}).")

[experiment_runner] 56 combinaciones ya en C:\Users\valef\OneDrive\Desktop\Credit_Card_Project\results\experiments_master.csv o C:\Users\valef\OneDrive\Desktop\Credit_Card_Project\results\skipped_timeout_log.csv, se omiten (resume=True).
[experiment_runner] 40 combinaciones a ejecutar.


[experiment_runner] OK xgboost/adasyn/grid_search — f1=0.5366±0.0091 (459.6s)


[experiment_runner] OK xgboost/adasyn/random_search — f1=0.5255±0.0043 (134.8s)


[experiment_runner] OK xgboost/adasyn/bayesian_optuna — f1=0.5222±0.0092 (136.4s)


[experiment_runner] OK xgboost/adasyn/genetic_deap — f1=0.5206±0.0087 (414.4s)


...

[experiment_runner] OK knn/adasyn/genetic_deap — f1=0.4891±0.0046 (1997.1s)


[experiment_runner] OK knn/class_weight/grid_search — f1=0.5260±0.0058 (27.0s)


[experiment_runner] OK knn/class_weight/random_search — f1=0.5265±0.0050 (33.6s)


[experiment_runner] OK knn/class_weight/bayesian_optuna — f1=0.5266±0.0051 (1813.6s)


[experiment_runner] OK knn/class_weight/genetic_deap — f1=0.5238±0.0039 (3995.8s)

40 combinaciones nuevas corridas en esta sesion.
96 filas totales en la tabla maestra (C:\Users\valef\OneDrive\Desktop\Credit_Card_Project\results\experiments_master.csv).


## Resultado

Tabla de métricas (media ± desviación estándar del outer loop, nunca un valor puntual), ordenada por la métrica principal de validación (ver `config.DEFAULT_SCORING` y la justificación en `evaluation.py`).

In [5]:
metric_for_ranking = config.DEFAULT_SCORING
table_sorted = table.sort_values(f"{metric_for_ranking}_mean", ascending=False)
format_metrics_table(table_sorted)

,model,technique,method,accuracy,precision,recall,f1,roc_auc
79,random_forest,class_weight,genetic_deap,0.7776 ± 0.0053,0.498 ± 0.0098,0.6042 ± 0.0092,0.546 ± 0.0088,0.7813 ± 0.0056
77,random_forest,class_weight,random_search,0.7759 ± 0.007,0.4951 ± 0.0129,0.6071 ± 0.006,0.5453 ± 0.0088,0.782 ± 0.0045
62,xgboost,class_weight,bayesian_optuna,0.7726 ± 0.0094,0.4896 ± 0.0161,0.6154 ± 0.0089,0.5451 ± 0.0099,0.7817 ± 0.0041
76,random_forest,class_weight,grid_search,0.7743 ± 0.004,0.492 ± 0.0072,0.6103 ± 0.0112,0.5448 ± 0.0074,0.7833 ± 0.0039
78,random_forest,class_weight,bayesian_optuna,0.7785 ± 0.0068,0.4998 ± 0.0126,0.598 ± 0.0035,0.5444 ± 0.0079,0.7804 ± 0.0055
...,...,...,...,...,...,...,...,...
82,knn,none,bayesian_optuna,0.8023 ± 0.0028,0.5924 ± 0.0134,0.3418 ± 0.0019,0.4334 ± 0.0038,0.7318 ± 0.0146
19,logistic_regression,none,genetic_deap,0.8071 ± 0.0017,0.6599 ± 0.0114,0.2647 ± 0.0024,0.3778 ± 0.0037,0.75 ± 0.0096
16,logistic_regression,none,grid_search,0.8071 ± 0.0017,0.6597 ± 0.0113,0.2647 ± 0.0024,0.3778 ± 0.0037,0.75 ± 0.0096
17,logistic_regression,none,random_search,0.8071 ± 0.0017,0.6597 ± 0.0113,0.2647 ± 0.0024,0.3778 ± 0.0037,0.75 ± 0.0096


Continuar en `05_results_analysis.ipynb`: funciona directamente desde `results/experiments_master.csv`, en una sesión nueva, sin necesitar nada de este notebook en memoria.